# Passerelle — expériences M1 à M3

Ce notebook pilote le code de `passerelle/`. Les fonctions HTTP, SQLite et mémoire
ne sont plus redéfinies entre les expériences. Installation : voir `README.md`.

**Exécuter de haut en bas.** Les appels au modèle sont désactivés par défaut.
Les sorties des anciennes expériences sont disponibles dans l'historique Git.
Le notebook initial fourni est conservé dans `archive/test_original.ipynb`.

## Configuration

Modifier ici le modèle et les paramètres. Pour continuer sans Ollama, laisser
`RUN_LLM = False`. La base de démonstration est locale et ignorée par Git.
Après modification d'un module Python, redémarrer le kernel et relancer les cellules.

In [ ]:
from uuid import uuid4

from passerelle import (
    SYSTEM_PROMPT,
    Assistant,
    ContextManager,
    OllamaClient,
    Settings,
    SQLiteStore,
    build_context,
)

RUN_LLM = False
RUN_BENCHMARK = False
settings = Settings(model="qwen2.5:3b", num_ctx=2048, max_tokens=128)
client = OllamaClient(settings)
store = SQLiteStore(settings.db_path)
assistant = Assistant(client, store, max_pairs=2)
demo_id = f"notebook-{uuid4().hex[:8]}"
print("Base :", store.path)
print("Conversation de démonstration :", demo_id)


def display_fragment(fragment):
    print(fragment, end="", flush=True)

## M1 — une réponse en streaming

`result.content` contient la réponse complète ; le callback affiche les fragments.
Les métriques restent dans `result.metrics`. Aucun historique global n'est modifié.

In [ ]:
if RUN_LLM:
    result = client.generate(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Explique en deux phrases ce qu'est un CPU."},
        ],
        stream=True,
        on_token=display_fragment,
    )
    print("\n", result.metrics.as_dict())
else:
    print("Appel ignoré : activer RUN_LLM pour utiliser Ollama.")

## M2 — contexte et mesures à chaud

Expérience facultative : une chauffe puis une mesure pour chaque capacité de contexte.
Même question et même plafond de sortie ; relever les tokens réellement générés.
Le délai du premier texte est mesuré côté client, les autres phases côté serveur.
Une seule mesure ne permet pas de conclure sur la performance générale d'un réglage.

In [ ]:
if RUN_LLM and RUN_BENCHMARK:
    question = [{"role": "user", "content": "Explique en deux phrases ce qu'est un CPU."}]
    measurements = []
    for context_size in (512, 2048):
        client.generate(question, num_ctx=context_size, max_tokens=64)
        measured = client.generate(question, stream=True, num_ctx=context_size, max_tokens=64)
        measurements.append({"num_ctx": context_size, **measured.metrics.as_dict()})
    for measurement in measurements:
        print(measurement)
else:
    print("Benchmark ignoré : activer RUN_LLM et RUN_BENCHMARK.")

## M3 — persistance et séparation des conversations, sans LLM

Ces écritures synthétiques peuvent être rejouées : elles remplacent les mêmes lignes.
Une nouvelle instance du stockage retrouve les données ; chaque lecture filtre
sur **l'utilisateur et la conversation**. L'authentification sera ajoutée plus tard.

In [ ]:
for user, name in [("alice", "Lina"), ("bob", "Karim")]:
    store.save(
        user,
        "storage-demo",
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Je m'appelle {name}."},
            {"role": "assistant", "content": "Bien reçu."},
        ],
    )
reopened = SQLiteStore(store.path)
for user in ("alice", "bob"):
    print(user, reopened.load(user, "storage-demo")[1]["content"])
print("Conversation absente :", reopened.load("alice", "unknown"))

## M3 — historique complet et fenêtre récente

La sélection ne modifie pas l'historique. Avec un seul couple récent, le prénom
et la contrainte présents au début ne sont plus transmis. Limiter les couples
ne garantit pas une limite en tokens.

In [ ]:
history = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Je m'appelle Lina et je travaille en local sans GPU."},
    {"role": "assistant", "content": "Bien reçu."},
    {"role": "user", "content": "Réponds en français."},
    {"role": "assistant", "content": "Entendu."},
    {"role": "user", "content": "Quel est mon prénom et ma contrainte matérielle ?"},
]
recent = build_context(history, max_pairs=1)
print("Historique complet :", len(history), "messages")
print("Contexte récent :", len(recent), "messages")
recent

## M3 — comparaison sans résumé / avec résumé

Le même modèle réalise un appel interne de résumé, puis l'appel conversationnel.
Le résumé est une donnée dérivée, pas une nouvelle instruction système. Vérifier
la présence de Lina et de la contrainte CPU/local, ainsi que l'absence d'invention.
Un résumé vide ou tronqué produit une erreur explicite.

In [ ]:
if RUN_LLM:
    memory = ContextManager(client, max_pairs=1)
    for mode in ("window", "summary"):
        prepared = memory.prepare(history, mode=mode)
        print(f"\nMode {mode} | Résumé : {prepared.summary}")
        comparison = client.generate(prepared.messages, stream=True, on_token=display_fragment)
        print("\n", comparison.metrics.as_dict())
else:
    print("Comparaison LLM ignorée ; la sélection précédente est vérifiable hors ligne.")

## M3 — conversation complète persistante

`assistant.chat` charge, prépare le contexte, génère, puis sauvegarde le tour complet.
Réexécuter cette cellule ajoute de nouveaux tours. Pour reprendre après un
redémarrage, conserver l'identifiant `demo_id` affiché en configuration.

In [ ]:
if RUN_LLM:
    for text in ("Je m'appelle Lina.", "Comment je m'appelle ?"):
        turn = assistant.chat("alice", demo_id, text, on_token=display_fragment)
        print("\n", turn.metrics.as_dict())
    saved = store.load("alice", demo_id)
    print("Messages originaux sauvegardés :", len(saved))
else:
    print("Conversation LLM ignorée. Le stockage SQLite a été vérifié plus haut.")

## Suite

- Ajouter un budget en tokens avant les appels (y compris celui de résumé).
- Mettre le résumé en cache avec la position des messages déjà couverts.
- Évaluer les résumés et réponses sur plusieurs conversations ; conserver les erreurs.
- Introduire les documents et leur provenance au module M4.

Les identités de ce TP sont simulées. Les tests logiciels ne prouvent ni la
fidélité du modèle, ni une isolation de sécurité prête pour la production.